In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 4.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.1/381.1 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 771.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 688.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("unsloth/Phi-3-mini-4k-instruct-bnb-4bit", token="<your-hf-token>") # Replace with your actual Hugging Face token if needed

# NOTE: Removed tokenizer.chat_template and tokenizer.add_special_tokens as per user request to not use Jinja templates.
# The `from_pretrained` method should load the correct special tokens for Phi-3.

with open("coffee_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

ds = Dataset.from_list(data)

def format_chat_template(examples):
    texts = []
    for prompt, response in zip(examples["prompt"], examples["response"]):
        # Manually construct the string using Phi-3's chat format:
        # <s><|user|>
        # {prompt}<|end|>
        # <|assistant|>
        # {response}<|endoftext|>
        formatted_string = (
            tokenizer.bos_token +
            "<|user|>\n" + prompt + "<|end|>\n" +
            "<|assistant|>\n" + response + tokenizer.eos_token
        )
        texts.append(formatted_string)
    return {"text": texts}

# Apply formatting to the dataset
dataset = ds.map(
    format_chat_template,
    batched=True,
    remove_columns=ds.column_names,
)

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

In [ ]:
from unsloth.models import FastLanguageModel

# Load the base model first
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit", # Choose your model name here
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    token = "<your-hf-token>", # Use the same Hugging Face token as before
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # Reduced rank for smaller dataset - better generalization
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,  # Reduced from 128 (rank*2) to prevent overfitting
    lora_   dropout = 0.05,  # Small dropout for regularization
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 42,
    max_seq_length = 2048,
)

==((====))==  Unsloth 2026.1.2: Fast Mistral patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.1.2 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch # Import torch library

training_args = TrainingArguments(
    output_dir = "./coffee_assistant_model",
    num_train_epochs = 5,  # Increased epochs for better learning
    per_device_train_batch_size = 2,  # Kept small for memory constraints
    gradient_accumulation_steps = 4,
    warmup_steps = 20,
    logging_steps = 5,
    save_strategy = "epoch",
    # evaluation_strategy = "no", # Removed to resolve TypeError
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 42,
    report_to = "none",
    max_grad_norm = 0.3,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = training_args,
    packing = False,  # Set to False for chat format
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/71 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 71 | Num Epochs = 5 | Total steps = 45
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Step,Training Loss
5,2.514100
10,2.303500
15,2.105900
20,1.778700
25,1.446600
30,1.353100
35,1.145100
40,0.914600
45,0.892400


TrainOutput(global_step=45, training_loss=1.605998855166965, metrics={'train_runtime': 138.775, 'train_samples_per_second': 2.558, 'train_steps_per_second': 0.324, 'total_flos': 943436833044480.0, 'train_loss': 1.605998855166965, 'epoch': 5.0})

In [ ]:
model.save_pretrained("coffee_assistant_lora")
tokenizer.save_pretrained("coffee_assistant_lora")

('coffee_assistant_lora/tokenizer_config.json',
 'coffee_assistant_lora/special_tokens_map.json',
 'coffee_assistant_lora/chat_template.jinja',
 'coffee_assistant_lora/tokenizer.model',
 'coffee_assistant_lora/added_tokens.json',
 'coffee_assistant_lora/tokenizer.json')

In [ ]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
 

In [ ]:
test_queries = [
    "What is in your Mysore Cappuccino?",
    "How do I make cold brew at home?",
    "What's your return policy?",
]

for query in test_queries:
    messages = [{"role": "user", "content": query}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Query: {query}")
    print(f"Response: {response}")
    print("-" * 50)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Query: What is in your Mysore Cappuccino?
Response: What is in your Mysore Cappuccino? Our signature 'Masala Latte' (cappuccino) with spiced espresso, steamed milk infused with cardamom and nutmeg. Add a pinch of saffron or serve it ghee-side for an authentic experience!
--------------------------------------------------
Query: How do I make cold brew at home?
Response: How do I make cold brew at home? Our easy guide: 1) Grind coarse (filter coffee). 2) Mix with water in 1:8 ratio. 3) Steep for 16-24 hours in room temp or fridge. 4) Strain through cheesecloth/coffee sock, squeeze gently to extract all liquid. We sell premium stainless steel filter bags that double as French press handles! Cold brew concentrate can be diluted with equal parts milk and hot water, sweetened with honey syrup, served over ice. It lasts up to two weeks refrigerated without oxygen absorbers.
--------------------------------------------------
Query: What's your return policy?
Response: What's your return polic

In [ ]:
from unsloth import is_bfloat16_supported

In [ ]:
model.save_pretrained_gguf(
    "coffee_assistant_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

print("Model saved in GGUF format for Ollama!")
print("To use with Ollama, create a Modelfile with:")
print("""
FROM ./coffee_assistant_gguf/coffee_assistant_gguf-Q4_K_M.gguf
TEMPLATE """ + """\"{{ if .System }}<|system|>
{{ .System }}<|end|>
{{ end }}{{ if .Prompt }}<|user|>
{{ .Prompt }}<|end|>
{{ end }}<|assistant|>
{{ .Response }}<|end|>\"\"""")
print("\nThen run: ollama create coffee-assistant -f Modelfile")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [02:32<02:32, 152.47s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:59<00:00, 119.74s/it]


Unsloth: Merge process complete. Saved to `/content/coffee_assistant_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['phi-3-mini-4k-instruct.F16.gg